# Code Execution with Files API

This notebook demonstrates an end-to-end analytics workflow using Anthropic code execution tools: upload a dataset, run an analysis prompt, and retrieve generated outputs.

In [8]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
from pathlib import Path

load_dotenv()

client = Anthropic(
    default_headers={
        "anthropic-beta": "code-execution-2025-08-25, files-api-2025-04-14"
    }
)
model = "claude-sonnet-4-5-20250929"

## Helper Utilities and File Operations

Define reusable chat helpers plus file-management utilities for upload, listing, metadata lookup, download, and deletion.

In [12]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=2000,
):
    params = {
        "model": model,
        "max_tokens": 10000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])


def upload(file_path):
    path = Path(file_path)
    extension = path.suffix.lower()

    mime_type_map = {
        ".pdf": "application/pdf",
        ".txt": "text/plain",
        ".md": "text/plain",
        ".py": "text/plain",
        ".js": "text/plain",
        ".html": "text/plain",
        ".css": "text/plain",
        ".csv": "text/csv",
        ".json": "application/json",
        ".xml": "application/xml",
        ".xlsx": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        ".xls": "application/vnd.ms-excel",
        ".jpeg": "image/jpeg",
        ".jpg": "image/jpeg",
        ".png": "image/png",
        ".gif": "image/gif",
        ".webp": "image/webp",
    }

    mime_type = mime_type_map.get(extension)

    if not mime_type:
        raise ValueError(f"Unknown mimetype for extension: {extension}")
    filename = path.name

    with open(file_path, "rb") as file:
        return client.beta.files.upload(file=(filename, file, mime_type))


def list_files():
    return client.beta.files.list()


def delete_file(id):
    return client.beta.files.delete(id)


def download_file(id, filename=None):
    file_content = client.beta.files.download(id)

    if not filename:
        file_metadata = get_metadata(id)
        file_content.write_to_file(file_metadata.filename)
    else:
        file_content.write_to_file(filename)


def get_metadata(id):
    return client.beta.files.retrieve_metadata(id)

## Upload Dataset for Analysis

Resolve the CSV path robustly across notebook working directories, then upload it to the Files API and capture file metadata.

In [10]:
from pathlib import Path

csv_candidates = [Path("data/streaming.csv"), Path("../data/streaming.csv")]
csv_path = next((p for p in csv_candidates if p.exists() and p.is_file()), None)

if csv_path is None:
    searched = ", ".join(str(p) for p in csv_candidates)
    raise FileNotFoundError(f"streaming.csv not found. Searched: {searched}")

file_metadata = upload(str(csv_path))
file_metadata

FileMetadata(id='file_011CbpdCQGPyj7hzyZiMPHsS', created_at=datetime.datetime(2026, 6, 7, 21, 40, 10, 576000, tzinfo=datetime.timezone.utc), filename='streaming.csv', mime_type='text/csv', size_bytes=25733, type='file', downloadable=False, scope=None)

## Run Code Execution Analysis

Send an analysis request that includes the uploaded file and invokes the code execution tool to compute insights and generate visual output.

In [13]:
messages = []

add_user_message(
    messages,
    [
        {
            "type": "text",
            "text": """
Run a detailed analysis to determine major drivers of churn.
Your final output should include at least one detailed plot summarizing your findings.

Critical note: Every time you execute code, you're starting with a completely clean slate. 
No variables or library imports from previous executions exist. You need to redeclare/reimport all variables/libraries.
            """,
        },
        {"type": "container_upload", "file_id": file_metadata.id},
    ],
)

chat(messages, tools=[{"type": "code_execution_20250825", "name": "code_execution"}])

Message(id='msg_01RLUrL2cVnAXxgRPokpsXzE', container=Container(id='container_011CbpdaqxLwZVrNDGwyyD6f', expires_at=datetime.datetime(2026, 6, 7, 22, 45, 18, 400362, tzinfo=TzInfo(0))), content=[TextBlock(citations=None, text="I'll run a focused churn analysis on your streaming.csv file. Let me start by exploring the data and then provide the requested insights.", type='text'), ServerToolUseBlock(id='srvtoolu_01F3jqahGRY5HYuQxPG43XC4', caller=None, input={'command': 'cd $INPUT_DIR && head -20 streaming.csv'}, name='bash_code_execution', type='server_tool_use'), BashCodeExecutionToolResultBlock(content=BashCodeExecutionResultBlock(content=[], return_code=0, stderr='', stdout='UserID,SubscriptionTier,TotalViewingHoursLastMonth,TopGenre,BingeWatchingSessionsLastMonth,NumberOfUniqueTitlesWatchedLastMonth,AverageSessionDurationMinutes,CustomerServiceInteractionsLastYear,MonthlyCost,Churned\nUSER_00001,Basic,47.9,Comedy,5,15,32.6,3,7.99,0\nUSER_00002,Premium,41.4,Drama,5,9,45.7,3,17.99,0\nUSE

## Download Generated Output

Download a generated artifact from the Files API using its file ID so you can inspect or share the analysis output locally.

In [ ]:
download_file("file_011CPYZqxoMSsfbrSzFw8j9X")